In [13]:
# ============================================
# Setup the Jupyter version of Dash
# ============================================
from jupyter_dash import JupyterDash

import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output
import base64

JupyterDash.infer_jupyter_proxy_config()

# ============================================
# Utilities
# ============================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ============================================
# Import CRUD Module (Project One)
# ============================================
from CRUD_Python_Module import AnimalShelterCRUD

# ============================================
# Database Connection
# ============================================
username = "aacuser"
password = "ChooseAPassword123"

db = AnimalShelterCRUD(username, password)

# Retrieve all data
df = pd.DataFrame.from_records(db.read({}))
df.drop(columns=['_id'], inplace=True)

# ============================================
# MongoDB Queries for Rescue Types
# ============================================
water_query = {
    "animal_type": "Dog",
    "breed": {"$in": [
        "Labrador Retriever Mix",
        "Chesapeake Bay Retriever",
        "Newfoundland"
    ]},
    "sex_upon_outcome": "Intact Female",
    "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
}

mountain_query = {
    "animal_type": "Dog",
    "breed": {"$in": [
        "German Shepherd",
        "Alaskan Malamute",
        "Old English Sheepdog",
        "Siberian Husky",
        "Rottweiler"
    ]},
    "sex_upon_outcome": "Intact Male",
    "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
}

disaster_query = {
    "animal_type": "Dog",
    "breed": {"$in": [
        "Doberman Pinscher",
        "German Shepherd",
        "Golden Retriever",
        "Bloodhound",
        "Rottweiler"
    ]},
    "sex_upon_outcome": "Intact Male",
    "age_upon_outcome_in_weeks": {"$gte": 20, "$lte": 300}
}

# ============================================
# Dashboard Layout
# ============================================
app = JupyterDash(__name__)

# Load Grazioso Salvare logo
image_filename = 'Grazioso Salvare Logo.png'
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

app.layout = html.Div([

    html.Center([
        html.Img(
            src='data:image/png;base64,{}'.format(encoded_image.decode()),
            style={'height': '100px'}
        ),
        html.H2("Grazioso Salvare Animal Rescue Dashboard"),
        html.H6("Dashboard by Lacey")
    ]),

    html.Hr(),

    html.Div([
        html.Label("Select Rescue Type"),
        dcc.RadioItems(
            id='filter-type',
            options=[
                {'label': 'Water Rescue', 'value': 'water'},
                {'label': 'Mountain or Wilderness Rescue', 'value': 'mountain'},
                {'label': 'Disaster or Individual Tracking', 'value': 'disaster'},
                {'label': 'Reset', 'value': 'reset'}
            ],
            value='reset'
        )
    ]),

    html.Hr(),

    dash_table.DataTable(
        id='datatable-id',
        columns=[{"name": i, "id": i, "selectable": True} for i in df.columns],
        data=df.to_dict('records'),
        page_size=10,
        sort_action="native",
        filter_action="native",
        row_selectable="single",
        selected_rows=[]
    ),

    html.Br(),
    html.Hr(),

    html.Div(
        style={'display': 'flex'},
        children=[
            html.Div(id='graph-id', style={'width': '50%'}),
            html.Div(id='map-id', style={'width': '50%'})
        ]
    )
])

# ============================================
# Callbacks
# ============================================

# Update Data Table
@app.callback(
    Output('datatable-id', 'data'),
    Input('filter-type', 'value')
)
def update_dashboard(filter_type):

    if filter_type == 'water':
        data = db.read(water_query)
    elif filter_type == 'mountain':
        data = db.read(mountain_query)
    elif filter_type == 'disaster':
        data = db.read(disaster_query)
    else:
        data = db.read({})

    df_filtered = pd.DataFrame.from_records(data)
    df_filtered.drop(columns=['_id'], inplace=True)

    return df_filtered.to_dict('records')


# Update Pie Chart (FIXED NoneType error)
@app.callback(
    Output('graph-id', "children"),
    Input('datatable-id', "derived_virtual_data")
)
def update_graphs(viewData):

    if viewData is None or len(viewData) == 0:
        return []

    dff = pd.DataFrame.from_dict(viewData)

    fig = px.pie(
        dff,
        names='breed',
        title='Breed Distribution'
    )

    return [dcc.Graph(figure=fig)]


# Highlight Selected Column
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    Input('datatable-id', 'selected_columns')
)
def update_styles(selected_columns):
    if selected_columns is None:
        return []
    return [{
        'if': {'column_id': i},
        'background_color': '#D2F3FF'
    } for i in selected_columns]


# Update Map (FIXED NoneType error)
@app.callback(
    Output('map-id', "children"),
    [
        Input('datatable-id', "derived_virtual_data"),
        Input('datatable-id', "derived_virtual_selected_rows")
    ]
)
def update_map(viewData, index):

    if viewData is None or index is None or len(index) == 0:
        return []

    dff = pd.DataFrame.from_dict(viewData)
    row = index[0]

    return [
        dl.Map(
            style={'width': '100%', 'height': '500px'},
            center=[30.75, -97.48],
            zoom=10,
            children=[
                dl.TileLayer(),
                dl.Marker(
                    position=[dff.iloc[row, 13], dff.iloc[row, 14]],
                    children=[
                        dl.Tooltip(dff.iloc[row, 4]),
                        dl.Popup([
                            html.H4("Animal Name"),
                            html.P(dff.iloc[row, 9])
                        ])
                    ]
                )
            ]
        )
    ]


# ============================================
# Run Server
# ============================================
app.run_server()


Dash app running on https://loveregion-lightgreen-3000.codio.io/proxy/8050/
